## LOPO

### Fitbit + EMA

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score
# time window = 2 - 7 days
time_window = 2
df_fitbit = pd.read_csv(f'results_fitbit_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_ema    = pd.read_csv(f'results_ema_{time_window}_LOPO.csv', parse_dates=['start_date'])

In [3]:
# Rename prediction and probability columns to distinguish modalities
df_fitbit.rename(columns={'pred': 'fitbit_pred', 'prob': 'fitbit_prob'}, inplace=True)
df_ema.rename(columns={'pred': 'ema_pred', 'prob': 'ema_prob'}, inplace=True)

# Merge the two dataframes, keeping the true label column from Fitbit
df = df_fitbit.merge(
    df_ema[['PID', 'start_date', 'ema_pred', 'ema_prob']], 
    on=['PID', 'start_date'], how='inner'
)

# === 2. Baseline Soft Voting Evaluation ===
# Equal-weighted soft voting
df['soft_vote_prob'] = 0.5 * df['fitbit_prob'] + 0.5 * df['ema_prob']
df['soft_vote_pred'] = (df['soft_vote_prob'] >= 0.5).astype(int)

# Compute macro-balanced accuracy and macro-F1 by averaging across participants
baseline_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred'])
).mean()
baseline_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred'])
).mean()

print(f"Baseline Soft Voting → Macro BA: {baseline_macro_ba:.4f}, Macro F1: {baseline_macro_f1:.4f}")

# === 3. Leave-One-Participant-Out (LOPO) Weight Optimization ===
pids = df['PID'].unique()
df['soft_vote_prob_opt'] = np.nan
df['soft_vote_pred_opt'] = np.nan

for pid_test in pids:
    is_test = df['PID'] == pid_test
    train_idx = df[~is_test].index
    val_idx = df[is_test].index

    # Extract training probabilities and true labels
    fitbit_tr = df.loc[train_idx, 'fitbit_prob'].values
    ema_tr = df.loc[train_idx, 'ema_prob'].values
    y_tr = df.loc[train_idx, 'true']

    # Grid search to find the optimal weight w for Fitbit
    best_w, best_score = 0.5, -np.inf
    for w in np.linspace(0, 1, 101):
        prob_tr = w * fitbit_tr + (1 - w) * ema_tr
        df_tr = df.loc[train_idx].copy()
        df_tr['prob_w'] = prob_tr
        df_tr['pred_w'] = (prob_tr >= 0.5).astype(int)

        # Compute macro-balanced accuracy across PIDs
        score = df_tr.groupby('PID').apply(
            lambda g: balanced_accuracy_score(g['true'], g['pred_w'])
        ).mean()

        if score > best_score:
            best_score, best_w = score, w

    # Use the best weight to predict on the held-out test set
    fitbit_val = df.loc[val_idx, 'fitbit_prob'].values
    ema_val = df.loc[val_idx, 'ema_prob'].values
    prob_val = best_w * fitbit_val + (1 - best_w) * ema_val

    df.loc[val_idx, 'soft_vote_prob_opt'] = prob_val
    df.loc[val_idx, 'soft_vote_pred_opt'] = (prob_val >= 0.5).astype(int)

    # print(f"LOPO fold PID={pid_test}: best w={best_w:.2f}, train-macro-BA={best_score:.4f}")

# === 4. Final Evaluation of Optimized Soft Voting ===
opt_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred_opt'])
).mean()
opt_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred_opt'])
).mean()

print(f"\nOptimized Soft Voting → Macro BA: {opt_macro_ba:.4f}, Macro F1: {opt_macro_f1:.4f}")


Baseline Soft Voting → Macro BA: 0.6485, Macro F1: 0.6839

Optimized Soft Voting → Macro BA: 0.6277, Macro F1: 0.6516


### Fitbit + MEMS

In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score
# time window = 2 - 7 days
time_window = 2
df_fitbit = pd.read_csv(f'results_fitbit_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_mems    = pd.read_csv(f'results_mems_{time_window}_LOPO.csv', parse_dates=['start_date'])

In [14]:
# Rename prediction and probability columns to distinguish modalities
df_fitbit.rename(columns={'pred': 'fitbit_pred', 'prob': 'fitbit_prob'}, inplace=True)
df_mems.rename(columns={'pred': 'mems_pred', 'prob': 'mems_prob'}, inplace=True)

# Merge the two dataframes, keeping the true label column from Fitbit
df = df_fitbit.merge(
    df_mems[['PID', 'start_date', 'mems_pred', 'mems_prob']], 
    on=['PID', 'start_date'], how='inner'
)

# === 2. Baseline Soft Voting Evaluation ===
# Equal-weighted soft voting
df['soft_vote_prob'] = 0.5 * df['fitbit_prob'] + 0.5 * df['mems_prob']
df['soft_vote_pred'] = (df['soft_vote_prob'] >= 0.5).astype(int)

# Compute macro-balanced accuracy and macro-F1 by averaging across participants
baseline_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred'])
).mean()
baseline_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred'])
).mean()

print(f"Baseline Soft Voting → Macro BA: {baseline_macro_ba:.4f}, Macro F1: {baseline_macro_f1:.4f}")

# === 3. Leave-One-Participant-Out (LOPO) Weight Optimization ===
pids = df['PID'].unique()
df['soft_vote_prob_opt'] = np.nan
df['soft_vote_pred_opt'] = np.nan

for pid_test in pids:
    is_test = df['PID'] == pid_test
    train_idx = df[~is_test].index
    val_idx = df[is_test].index

    # Extract training probabilities and true labels
    fitbit_tr = df.loc[train_idx, 'fitbit_prob'].values
    mems_tr = df.loc[train_idx, 'mems_prob'].values
    y_tr = df.loc[train_idx, 'true']

    # Grid search to find the optimal weight w for Fitbit
    best_w, best_score = 0.5, -np.inf
    for w in np.linspace(0, 1, 101):
        prob_tr = w * fitbit_tr + (1 - w) * mems_tr
        df_tr = df.loc[train_idx].copy()
        df_tr['prob_w'] = prob_tr
        df_tr['pred_w'] = (prob_tr >= 0.5).astype(int)

        # Compute macro-balanced accuracy across PIDs
        score = df_tr.groupby('PID').apply(
            lambda g: balanced_accuracy_score(g['true'], g['pred_w'])
        ).mean()

        if score > best_score:
            best_score, best_w = score, w

    # Use the best weight to predict on the held-out test set
    fitbit_val = df.loc[val_idx, 'fitbit_prob'].values
    mems_val = df.loc[val_idx, 'mems_prob'].values
    prob_val = best_w * fitbit_val + (1 - best_w) * mems_val

    df.loc[val_idx, 'soft_vote_prob_opt'] = prob_val
    df.loc[val_idx, 'soft_vote_pred_opt'] = (prob_val >= 0.5).astype(int)

    # print(f"LOPO fold PID={pid_test}: best w={best_w:.2f}, train-macro-BA={best_score:.4f}")

# === 4. Final Evaluation of Optimized Soft Voting ===
opt_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred_opt'])
).mean()
opt_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred_opt'])
).mean()

print(f"\nOptimized Soft Voting → Macro BA: {opt_macro_ba:.4f}, Macro F1: {opt_macro_f1:.4f}")


Baseline Soft Voting → Macro BA: 0.7831, Macro F1: 0.8480

Optimized Soft Voting → Macro BA: 0.7710, Macro F1: 0.8484


### EMA + MEMS

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score
# time window = 2 - 7 days
time_window = 2
df_ema = pd.read_csv(f'results_ema_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_mems    = pd.read_csv(f'results_mems_{time_window}_LOPO.csv', parse_dates=['start_date'])

In [17]:
# Rename prediction and probability columns to distinguish modalities
df_ema.rename(columns={'pred': 'ema_pred', 'prob': 'ema_prob'}, inplace=True)
df_mems.rename(columns={'pred': 'mems_pred', 'prob': 'mems_prob'}, inplace=True)

# Merge the two dataframes, keeping the true label column from mems
df = df_mems.merge(
    df_ema[['PID', 'start_date', 'ema_pred', 'ema_prob']], 
    on=['PID', 'start_date'], how='inner'
)

# === 2. Baseline Soft Voting Evaluation ===
# Equal-weighted soft voting
df['soft_vote_prob'] = 0.5 * df['ema_prob'] + 0.5 * df['mems_prob']
df['soft_vote_pred'] = (df['soft_vote_prob'] >= 0.5).astype(int)

# Compute macro-balanced accuracy and macro-F1 by averaging across participants
baseline_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred'])
).mean()
baseline_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred'])
).mean()

print(f"Baseline Soft Voting → Macro BA: {baseline_macro_ba:.4f}, Macro F1: {baseline_macro_f1:.4f}")

# === 3. Leave-One-Participant-Out (LOPO) Weight Optimization ===
pids = df['PID'].unique()
df['soft_vote_prob_opt'] = np.nan
df['soft_vote_pred_opt'] = np.nan

for pid_test in pids:
    is_test = df['PID'] == pid_test
    train_idx = df[~is_test].index
    val_idx = df[is_test].index

    # Extract training probabilities and true labels
    ema_tr = df.loc[train_idx, 'ema_prob'].values
    mems_tr = df.loc[train_idx, 'mems_prob'].values
    y_tr = df.loc[train_idx, 'true']

    # Grid search to find the optimal weight w for ema
    best_w, best_score = 0.5, -np.inf
    for w in np.linspace(0, 1, 101):
        prob_tr = w * ema_tr + (1 - w) * mems_tr
        df_tr = df.loc[train_idx].copy()
        df_tr['prob_w'] = prob_tr
        df_tr['pred_w'] = (prob_tr >= 0.5).astype(int)

        # Compute macro-balanced accuracy across PIDs
        score = df_tr.groupby('PID').apply(
            lambda g: balanced_accuracy_score(g['true'], g['pred_w'])
        ).mean()

        if score > best_score:
            best_score, best_w = score, w

    # Use the best weight to predict on the held-out test set
    ema_val = df.loc[val_idx, 'ema_prob'].values
    mems_val = df.loc[val_idx, 'mems_prob'].values
    prob_val = best_w * ema_val + (1 - best_w) * mems_val

    df.loc[val_idx, 'soft_vote_prob_opt'] = prob_val
    df.loc[val_idx, 'soft_vote_pred_opt'] = (prob_val >= 0.5).astype(int)

    # print(f"LOPO fold PID={pid_test}: best w={best_w:.2f}, train-macro-BA={best_score:.4f}")

# === 4. Final Evaluation of Optimized Soft Voting ===
opt_macro_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_vote_pred_opt'])
).mean()
opt_macro_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_vote_pred_opt'])
).mean()

print(f"\nOptimized Soft Voting → Macro BA: {opt_macro_ba:.4f}, Macro F1: {opt_macro_f1:.4f}")


Baseline Soft Voting → Macro BA: 0.7209, Macro F1: 0.8314

Optimized Soft Voting → Macro BA: 0.7366, Macro F1: 0.8560


### Fitbit + EMA + MEMS

In [18]:
time_window = 2
df_fitbit = pd.read_csv(f'results_fitbit_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_ema = pd.read_csv(f'results_ema_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_mems    = pd.read_csv(f'results_mems_{time_window}_LOPO.csv', parse_dates=['start_date'])

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score

# === 1. Rename prediction and probability columns to distinguish modalities ===
df_fitbit.rename(columns={'pred': 'fitbit_pred', 'prob': 'fitbit_prob'}, inplace=True)
df_ema   .rename(columns={'pred': 'ema_pred',    'prob': 'ema_prob'   }, inplace=True)
df_mems  .rename(columns={'pred': 'mems_pred',   'prob': 'mems_prob'  }, inplace=True)

# Merge the three modality outputs: Fitbit + EMA first, then add MEMS
df = (
    df_fitbit
    .merge(df_ema[['PID', 'start_date', 'ema_pred', 'ema_prob']],
           on=['PID', 'start_date'], how='inner')
    .merge(df_mems[['PID', 'start_date', 'mems_pred', 'mems_prob']],
           on=['PID', 'start_date'], how='inner')
)

# === 2. Baseline Soft Voting with Equal Weights (1/3 each) ===
df['soft_prob_eq'] = (df['fitbit_prob'] + df['ema_prob'] + df['mems_prob']) / 3
df['soft_pred_eq'] = (df['soft_prob_eq'] >= 0.5).astype(int)

# Compute macro-balanced accuracy and macro-F1 by averaging across participants
baseline_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_pred_eq'])
).mean()
baseline_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_pred_eq'])
).mean()

print(f"Baseline (1/3 each) → Macro BA: {baseline_ba:.4f}, Macro F1: {baseline_f1:.4f}")

# === 3. LOPO: Grid Search for Optimal Weights (Fitbit, EMA, MEMS) ===
pids = df['PID'].unique()

# Initialize columns to store optimized predictions
df['soft_prob_opt'] = np.nan
df['soft_pred_opt'] = np.nan

# Leave-One-Participant-Out loop
for pid_test in pids:
    is_test   = df['PID'] == pid_test
    train_idx = df[~is_test].index
    val_idx   = df[ is_test].index

    # Extract modality probabilities and true labels for training set
    f_tr = df.loc[train_idx, 'fitbit_prob'].values
    e_tr = df.loc[train_idx, 'ema_prob'].values
    m_tr = df.loc[train_idx, 'mems_prob'].values
    y_tr = df.loc[train_idx, 'true'].values

    # Initialize search
    best_score = -np.inf
    best_w     = (1/3, 1/3, 1/3)

    # Coarse grid search over w1 (Fitbit), w2 (EMA), w3 = 1 - w1 - w2
    for w1 in np.linspace(0, 1, 21):
        for w2 in np.linspace(0, 1 - w1, 21):
            w3 = 1 - w1 - w2
            prob_tr = w1 * f_tr + w2 * e_tr + w3 * m_tr

            tmp = df.loc[train_idx].copy()
            tmp['prob_w'] = prob_tr
            tmp['pred_w'] = (prob_tr >= 0.5).astype(int)

            # Evaluate using macro-balanced accuracy
            score = tmp.groupby('PID').apply(
                lambda g: balanced_accuracy_score(g['true'], g['pred_w'])
            ).mean()

            if score > best_score:
                best_score, best_w = score, (w1, w2, w3)

    # Apply the best weights on the held-out test set
    f_val = df.loc[val_idx, 'fitbit_prob'].values
    e_val = df.loc[val_idx, 'ema_prob'].values
    m_val = df.loc[val_idx, 'mems_prob'].values
    w1, w2, w3 = best_w
    prob_val = w1 * f_val + w2 * e_val + w3 * m_val

    df.loc[val_idx, 'soft_prob_opt'] = prob_val
    df.loc[val_idx, 'soft_pred_opt'] = (prob_val >= 0.5).astype(int)

    # Optional: print best weight per fold
    # print(f"Fold LOPO PID={pid_test}: best_w={[round(w,2) for w in best_w]}, train-macro-BA={best_score:.4f}")

# === 4. Final Evaluation of Optimized Soft Voting ===
opt_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_pred_opt'])
).mean()
opt_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_pred_opt'])
).mean()

print(f"\nOptimized Soft Voting → Macro BA: {opt_ba:.4f}, Macro F1: {opt_f1:.4f}")


Baseline (1/3 each) → Macro BA: 0.8068, Macro F1: 0.8562

Optimized Soft Voting → Macro BA: 0.8123, Macro F1: 0.8826


### Fitbit + EMA + MEMS + Baseline

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

time_window = 2

# === 1. Read LSTM Output for Fitbit, EMA, and MEMS Modalities ===
df_fitbit = pd.read_csv(f'results_fitbit_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_ema    = pd.read_csv(f'results_ema_{time_window}_LOPO.csv', parse_dates=['start_date'])
df_mems   = pd.read_csv(f'results_mems_{time_window}_LOPO.csv', parse_dates=['start_date'])

# === 2. Read Baseline Survey Predictions ===
df_baseline = pd.read_csv("baseline_result.csv", parse_dates=['start_date', 'end_date'])

/tmp/ipykernel_54454/3030719493.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_baseline = pd.read_csv("baseline_result.csv", parse_dates=['start_date', 'end_date'])
/tmp/ipykernel_54454/3030719493.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_baseline = pd.read_csv("baseline_result.csv", parse_dates=['start_date', 'end_date'])


In [3]:
df_fitbit.rename(columns={'pred': 'fitbit_pred', 'prob': 'fitbit_prob'}, inplace=True)
df_ema   .rename(columns={'pred': 'ema_pred',    'prob': 'ema_prob'   }, inplace=True)
df_mems  .rename(columns={'pred': 'mems_pred',   'prob': 'mems_prob'  }, inplace=True)
# === 4. Merge All Three Modalities Based on PID and start_date ===
df = (
    df_fitbit
    .merge(df_ema[['PID', 'start_date', 'ema_pred', 'ema_prob']],  on=['PID', 'start_date'], how='inner')
    .merge(df_mems[['PID', 'start_date', 'mems_pred', 'mems_prob']], on=['PID', 'start_date'], how='inner')
)

# === 5. Initialize a Column for Baseline Probabilities ===
df['baseline_prob'] = np.nan

# === 6. Map Baseline Predictions into the Main DataFrame ===
# For each row in the merged dataframe, find the matching baseline prediction
# based on PID and if the start_date falls within the baseline interval
for idx, row in df.iterrows():
    pid, date = row['PID'], row['start_date']
    match = df_baseline[
        (df_baseline['PID'] == pid) &
        (df_baseline['start_date'] <= date) &
        (df_baseline['end_date'] >= date)
    ]
    if not match.empty:
        df.at[idx, 'baseline_prob'] = match.iloc[0]['pred_adherence']

In [4]:
# === Step 2: Equal-Weighted Soft Voting (Four Modalities) ===
# Average the prediction probabilities from all four modalities
df['soft_prob_eq'] = df[['fitbit_prob', 'ema_prob', 'mems_prob', 'baseline_prob']].mean(axis=1)
df['soft_pred_eq'] = (df['soft_prob_eq'] >= 0.5).astype(int)

# Evaluate macro balanced accuracy and macro F1-score by averaging across participants
baseline_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_pred_eq'])
).mean()
baseline_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_pred_eq'])
).mean()

print(f"Baseline Soft Voting (equal weights) → Macro BA: {baseline_ba:.4f}, Macro F1: {baseline_f1:.4f}")

# === Step 3: Optimized Soft Voting Weights via Leave-One-Participant-Out (LOPO) ===
pids = df['PID'].unique()

# Initialize columns to store optimized predictions
df['soft_prob_opt'] = np.nan
df['soft_pred_opt'] = np.nan

# Loop through each participant, treating them as the test fold
for pid_test in pids:
    is_test = df['PID'] == pid_test
    train_idx = df[~is_test].index
    val_idx   = df[ is_test].index

    # Extract training data probabilities and labels
    f_tr = df.loc[train_idx, 'fitbit_prob'].values
    e_tr = df.loc[train_idx, 'ema_prob'].values
    m_tr = df.loc[train_idx, 'mems_prob'].values
    b_tr = df.loc[train_idx, 'baseline_prob'].values
    y_tr = df.loc[train_idx, 'true'].values

    # Initialize best score and weights
    best_score = -np.inf
    best_w = (0.25, 0.25, 0.25, 0.25)

    # Grid search over possible weight combinations for the four modalities
    # w4 = 1 - w1 - w2 - w3 ensures weights sum to 1
    for w1 in np.linspace(0, 1, 11):
        for w2 in np.linspace(0, 1 - w1, 11):
            for w3 in np.linspace(0, 1 - w1 - w2, 11):
                w4 = 1 - w1 - w2 - w3
                prob_tr = w1 * f_tr + w2 * e_tr + w3 * m_tr + w4 * b_tr
                tmp = df.loc[train_idx].copy()
                tmp['prob_w'] = prob_tr
                tmp['pred_w'] = (prob_tr >= 0.5).astype(int)

                # Evaluate macro balanced accuracy
                score = tmp.groupby('PID').apply(
                    lambda g: balanced_accuracy_score(g['true'], g['pred_w'])
                ).mean()

                # Update best weights if better score is found
                if score > best_score:
                    best_score = score
                    best_w = (w1, w2, w3, w4)

    # Apply optimal weights to the held-out test participant
    f_val = df.loc[val_idx, 'fitbit_prob'].values
    e_val = df.loc[val_idx, 'ema_prob'].values
    m_val = df.loc[val_idx, 'mems_prob'].values
    b_val = df.loc[val_idx, 'baseline_prob'].values
    w1, w2, w3, w4 = best_w
    prob_val = w1 * f_val + w2 * e_val + w3 * m_val + w4 * b_val

    df.loc[val_idx, 'soft_prob_opt'] = prob_val
    df.loc[val_idx, 'soft_pred_opt'] = (prob_val >= 0.5).astype(int)

# === Step 4: Final Evaluation of Optimized Soft Voting ===
opt_ba = df.groupby('PID').apply(
    lambda g: balanced_accuracy_score(g['true'], g['soft_pred_opt'])
).mean()
opt_f1 = df.groupby('PID').apply(
    lambda g: f1_score(g['true'], g['soft_pred_opt'])
).mean()

print(f"\nOptimized Soft Voting (LOPO) → Macro BA: {opt_ba:.4f}, Macro F1: {opt_f1:.4f}")

Baseline Soft Voting (equal weights) → Macro BA: 0.7291, Macro F1: 0.8592

Optimized Soft Voting (LOPO) → Macro BA: 0.8287, Macro F1: 0.8845
